In [ ]:
# %% Cell 1: Install
!pip install connected-components-3d segmentation-models-pytorch --no-index --find-links=file:///kaggle/input/datasets/tylerde/my-pip-packages/

In [ ]:
# %% Cell 2: Setup
import sys
import os
import shutil
import torch

# Add external model code to path
sys.path.append("/kaggle/input/datasets/hengck23/hengck23-demo-submit-physionet")
sys.path.append("/kaggle/input/datasets/takashisomeya/physionet-final-submission-models")

# Add our src/ to path (uploaded as a Kaggle dataset or via utility script)
# Option A: if src/ is in the notebook's working directory
sys.path.append("/kaggle/working")
# Option B: if src/ is uploaded as a dataset
# sys.path.append("/kaggle/input/datasets/<user>/ecg-src")

from src.utils import load_config, load_test_metadata, read_image
from src.stage0 import run_stage0_parallel
from src.stage1 import run_stage1_parallel
from src.stage2 import run_stage2_parallel, compute_stage2_constants
from src.submission import make_submission

cfg = load_config("configs/default.yaml")
valid_df = load_test_metadata(cfg["paths"]["kaggle_dir"])
valid_ids = valid_df["id"].unique().tolist()
float_type = getattr(torch, cfg["inference"]["float_type"])

out_dir = cfg["paths"]["out_dir"]
os.makedirs(out_dir, exist_ok=True)

print(f"Test samples: {len(valid_ids)}")

In [ ]:
# %% Cell 3: Stage 0
read_img_fn = lambda sid: read_image(sid, cfg["paths"]["kaggle_dir"])
fail_0 = run_stage0_parallel(
    valid_ids, read_img_fn, out_dir, f"{cfg['paths']['hengck23_dir']}/weight", float_type
)

In [ ]:
# %% Cell 4: Stage 1
fail_1 = run_stage1_parallel(
    valid_ids, out_dir, f"{cfg['paths']['hengck23_dir']}/weight", fail_0, float_type
)

In [ ]:
# %% Cell 5: Stage 2
consts = compute_stage2_constants(cfg)
fail_2 = run_stage2_parallel(valid_ids, valid_df, cfg, consts, fail_1)

In [ ]:
# %% Cell 6: Submission
make_submission(valid_df, out_dir, "submission.csv")

In [ ]:
# %% Cell 7: Cleanup
shutil.rmtree(out_dir)
print("Done! Ready to submit submission.csv")